# Gopi TEC Brasília analysis - 09/27/2024

In [ ]:
#Download Gopi_TEC_BRAZ_09_27_2024_and_Dec_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip 'Gopi_TEC_BRAZ_09_27_2024_and_Dec_2024.zip'

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

pd.set_option('display.max_rows', None)

df = pd.read_csv('braz271-2024-09-27.Std', sep='\s+', header=None)

df.columns = ['time_ut', 'tec', 'tec_std', 'latitude']

base_date = datetime(2024, 9, 27)

def decimal_to_time(decimal_hours):
    hours = int(decimal_hours)
    minutes = int((decimal_hours - hours) * 60)
    seconds = int(((decimal_hours - hours) * 60 - minutes) * 60)
    return base_date + timedelta(hours=hours, minutes=minutes, seconds=seconds)

df['DATETIME'] = df['time_ut'].apply(decimal_to_time)

df['tec'] = pd.to_numeric(df['tec'].replace('-', np.nan))
df['tec_std'] = pd.to_numeric(df['tec_std'].replace('-', np.nan))

df = df[['DATETIME', 'tec', 'tec_std', 'latitude']]
df.columns = ['DATETIME', 'TEC', 'TEC_STD', 'LATITUDE']

print(df)

df.to_pickle('braz271-2024-09-27.pkl')

In [ ]:
df

In [ ]:
import pandas as pd
from datetime import time
import numpy as np

df['DATETIME'] = pd.to_datetime(df['DATETIME'])

target_times = [
    time(0, 50),
    time(1, 0),
    time(1, 10)
]

def find_closest_time(df, target_time):

    df_times = df['DATETIME'].dt.time


    target_seconds = target_time.hour * 3600 + target_time.minute * 60 + target_time.second


    df_seconds = (df['DATETIME'].dt.hour * 3600 +
                  df['DATETIME'].dt.minute * 60 +
                  df['DATETIME'].dt.second)


    time_diff = np.abs(df_seconds - target_seconds)


    closest_idx = time_diff.idxmin()

    return df.loc[[closest_idx]]

df_filtered_list = []
for target_time in target_times:
    closest_row = find_closest_time(df, target_time)
    df_filtered_list.append(closest_row)

df_filtered = pd.concat(df_filtered_list, ignore_index=True)

df_filtered = df_filtered.sort_values('DATETIME').reset_index(drop=True)

print("DataFrame filtered for closest times to 00:50, 01:00 and 01:10:")
print(df_filtered)

print("\nDifference from target times:")
for i, target_time in enumerate(target_times):
    actual_time = df_filtered.iloc[i]['DATETIME'].time()
    target_seconds = target_time.hour * 3600 + target_time.minute * 60 + target_time.second
    actual_seconds = actual_time.hour * 3600 + actual_time.minute * 60 + actual_time.second
    diff_seconds = abs(actual_seconds - target_seconds)
    print(f"Target: {target_time} | Actual: {actual_time} | Difference: {diff_seconds} seconds")

In [ ]:
reference_time = pd.Timestamp('2024-09-27 00:50:00')
wide_window = timedelta(minutes=5)
df_proximos = df[(df['DATETIME'] >= reference_time - wide_window) &
                 (df['DATETIME'] <= reference_time + wide_window)]
print("Records close to 00:50:")
print(df_proximos[['DATETIME', 'TEC']])

In [ ]:
print(df.iloc[[50, 60, 70]][['DATETIME', 'TEC']])

In [ ]:
df.to_pickle('braz271-2024-09-27_selected_0050_to_0110.pkl')

# EMBRACE

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Download TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
df_embrace_maps_2024 = pd.read_pickle('/content/TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
# df_embrace_maps_2024

In [ ]:
date = '2024-09-27'
times = ['00:50:00', '01:00:00', '01:10:00']

timestamps = [f'{date} {time}' for time in times]

result = df_embrace_maps_2024.loc[timestamps]

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)

In [ ]:
embrace_tec = Embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-15.94748, lon=-47.87787)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -15.94748
target_lon = -47.87787

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

In [ ]:
embrace_tec.tec_map = np_embrace_maps[1]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -15.94748
target_lon = -47.87787

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

In [ ]:
embrace_tec.tec_map = np_embrace_maps[2]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(embrace_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -15.94748
target_lon = -47.87787

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

# MAGGIA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
df_maggia_maps_2024 = pd.read_pickle('/content/TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_maggia_maps_2024['DATETIME'] = pd.to_datetime(df_maggia_maps_2024['DATETIME'])
df_maggia_maps_2024 = df_maggia_maps_2024.set_index('DATETIME')

In [ ]:
df_maggia_maps_2024.columns

In [ ]:
# df_maggia_maps_2024

In [ ]:
date = '2024-09-27'
times = ['00:50:00', '01:00:00', '01:10:00']

timestamps = [f'{date} {time}' for time in times]

result = df_maggia_maps_2024.loc[timestamps]

In [ ]:
result

In [ ]:
mapas_maggia = np.array(result.iloc[:]['TECMAP'])

In [ ]:
mapas_maggia

In [ ]:
np_maggia_maps = []
for i in range(len(mapas_maggia)):
    np_maggia_maps.append(mapas_maggia[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:

    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(tec_obj, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = tec_obj.extent
        lat_step, lon_step = tec_obj.lat_step, tec_obj.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, tec_obj.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, tec_obj.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, tec_obj.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, tec_obj.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points

class Maggia(TecMap):

    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)

In [ ]:
maggia_tec = Maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-15.94748, lon=-47.87787)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -15.94748
target_lon = -47.87787

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

In [ ]:
maggia_tec.tec_map = np_maggia_maps[1]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -15.94748
target_lon = -47.87787

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")

In [ ]:
maggia_tec.tec_map = np_maggia_maps[2]

print("=" * 60)
print("DETAILED BILINEAR INTERPOLATION ANALYSIS - MAGGIA")
print("=" * 60)

print("\n1. THE 4 CLOSEST POINTS:")
for i, point in enumerate(maggia_points):
    lat, lon, lat_idx, lon_idx = point
    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]
    print(f"   P{i+1}: ({lat:+6.1f}°, {lon:+6.1f}°) - Indexes: [{lat_idx:2d}, {lon_idx:2d}] - TEC: {tec_value:5.1f}")

target_lat = -15.94748
target_lon = -47.87787

print(f"\n2. TARGET POINT: ({target_lat:+6.5f}°, {target_lon:+6.5f}°)")

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

print(f"\n3. INTERPOLATION RECTANGLE BOUNDS:")
print(f"   Latitude:  {lat_min:+6.1f}° to {lat_max:+6.1f}° (range: {lat_max - lat_min:4.1f}°)")
print(f"   Longitude: {lon_min:+6.1f}° to {lon_max:+6.1f}° (range: {lon_max - lon_min:4.1f}°)")

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

print(f"\n4. TEC VALUES MAPPING:")
print(f"   f_00 (lat_min, lon_min) = ({lat_min:+6.1f}°, {lon_min:+6.1f}°) = {f_00:5.1f}")
print(f"   f_01 (lat_min, lon_max) = ({lat_min:+6.1f}°, {lon_max:+6.1f}°) = {f_01:5.1f}")
print(f"   f_10 (lat_max, lon_min) = ({lat_max:+6.1f}°, {lon_min:+6.1f}°) = {f_10:5.1f}")
print(f"   f_11 (lat_max, lon_max) = ({lat_max:+6.1f}°, {lon_max:+6.1f}°) = {f_11:5.1f}")

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

print(f"\n5. INTERPOLATION PARAMETERS:")
print(f"   t_x = (target_lon - lon_min) / (lon_max - lon_min)")
print(f"   t_x = ({target_lon:+6.5f} - ({lon_min:+6.1f})) / ({lon_max:+6.1f} - ({lon_min:+6.1f}))")
print(f"   t_x = {target_lon - lon_min:6.5f} / {lon_max - lon_min:4.1f} = {t_x:.6f}")
print(f"   ")
print(f"   t_y = (target_lat - lat_min) / (lat_max - lat_min)")
print(f"   t_y = ({target_lat:+6.5f} - ({lat_min:+6.1f})) / ({lat_max:+6.1f} - ({lat_min:+6.1f}))")
print(f"   t_y = {target_lat - lat_min:6.5f} / {lat_max - lat_min:4.1f} = {t_y:.6f}")

w_00 = (1 - t_x) * (1 - t_y)
w_01 = t_x * (1 - t_y)
w_10 = (1 - t_x) * t_y
w_11 = t_x * t_y

print(f"\n6. INTERPOLATION WEIGHTS:")
print(f"   w_00 = (1 - t_x) * (1 - t_y) = (1 - {t_x:.6f}) * (1 - {t_y:.6f}) = {w_00:.6f}")
print(f"   w_01 = t_x * (1 - t_y)       = {t_x:.6f} * (1 - {t_y:.6f})       = {w_01:.6f}")
print(f"   w_10 = (1 - t_x) * t_y       = (1 - {t_x:.6f}) * {t_y:.6f}       = {w_10:.6f}")
print(f"   w_11 = t_x * t_y             = {t_x:.6f} * {t_y:.6f}             = {w_11:.6f}")
print(f"   Sum of weights = {w_00 + w_01 + w_10 + w_11:.6f} (should be 1.0)")

interpolated_value = w_00 * f_00 + w_01 * f_01 + w_10 * f_10 + w_11 * f_11

print(f"\n7. FINAL CALCULATION:")
print(f"   Value = w_00*f_00 + w_01*f_01 + w_10*f_10 + w_11*f_11")
print(f"   Value = {w_00:.6f}*{f_00:5.1f} + {w_01:.6f}*{f_01:5.1f} + {w_10:.6f}*{f_10:5.1f} + {w_11:.6f}*{f_11:5.1f}")
print(f"   Value = {w_00*f_00:8.4f} + {w_01*f_01:8.4f} + {w_10*f_10:8.4f} + {w_11*f_11:8.4f}")
print(f"   Value = {interpolated_value:.6f}")

print(f"\n8. TARGET POINT RELATIVE POSITION:")
print(f"   In X direction (longitude): {t_x*100:.2f}% of the way from {lon_min}° to {lon_max}°")
print(f"   In Y direction (latitude):  {t_y*100:.2f}% of the way from {lat_min}° to {lat_max}°")

print(f"\n{'='*60}")
print(f"FINAL RESULT: TEC = {interpolated_value:.4f}")
print(f"{'='*60}")